# Lecture 9: From dense to sparse linear algebra

## Plan of today's lecture

- Parallel computing in NLA: the matrix-by-vector product (matvec) as the model example
    - distributed memory, MPI, 1D vs 2D blocking
- Sparse matrices (with a separate plan)
    - formats, SpMV, locality, fast direct solvers (start)

## Large scale dense matrices

- A dense $n \times n$ matrix needs $n^2$ numbers: for $n = 10^6$ this is 8 TB in double precision. It simply does not fit in memory.

- Possible options:
    - the matrix is **structured** (e.g. block Toeplitz with Toeplitz blocks -- next lectures), then compressed storage is possible;
    - the matrix is unstructured: store it in **distributed memory**, i.e. split it between several machines;
    - [MPI (Message Passing Interface)](https://en.wikipedia.org/wiki/Message_Passing_Interface) -- the standard for programming distributed memory.

### Distributed memory and MPI

- Split the matrix into blocks, store the blocks on different machines (**processes**).
- Every machine has its **own address space** and cannot touch the memory of the others.
- Machines communicate only by **messages**: send/receive partial results.
- MPI is a library implementing these messages; the typical pattern is **SPMD** (Single Program, Multiple Data): the same program runs everywhere, the rank of the process decides which block it owns.
- Two parameters of the model machine:
    - $t_s$ -- **latency**: time to send one message (seconds),
    - $t_w$ -- **per-word time**: seconds to transmit one number (inverse bandwidth).

### Example: matrix-by-vector product

- Goal: compute $y = Ax$, but $A$ does not fit in the memory of one machine.
- Split $A$ into blocks, distribute the blocks over $p$ processes.
- Two classical strategies:
    - **1D blocking**: split only the rows of $A$;
    - **2D blocking**: split both rows and columns (a $\sqrt{p} \times \sqrt{p}$ grid of blocks).

#### Total time of the matvec with 1D blocking

- Each process has $n/p$ complete rows of $A$, produces $n/p$ entries of $y$ -- but needs the **whole** vector $x$ ($n$ entries).
- Computation: $2n^2/p$ flops.
- Communication: broadcast of $x$ to all processes (a tree of messages):

$$T_{1D} \approx \underbrace{\frac{2n^2}{p}}_{\text{computation}} + \underbrace{t_s \log p + t_w n}_{\text{broadcast of } x}.$$

- The data term $t_w n$ does **not** shrink with $p$ -- this is the bottleneck.

#### Total time of the matvec with 2D blocking

- Each process owns an $\frac{n}{\sqrt{p}} \times \frac{n}{\sqrt{p}}$ block and needs only $n/\sqrt{p}$ entries of $x$ (the piece owned by its column of processes).
- Computation: still $2n^2/p$ flops.
- Communication: broadcast $n/\sqrt{p}$ words along the process column + reduce (sum up) $n/\sqrt{p}$ partial results along the process row:

$$T_{2D} \approx \frac{2n^2}{p} + 2\big(t_s \log p + t_w n/\sqrt{p}\big).$$

- The data term shrinks by $\sqrt{p}$: this is exactly why 2D blocking wins at scale.

#### What the cost model says

- Plot the model times: computation, 1D, 2D.
- **Counterexample to 'just add processors'**: at large $p$ both curves flatten at the latency floor $t_s \log p$ -- more processes do not help anymore.

In [ ]:
n = 100_000                    # matrix size
p = 2.0 ** np.arange(0, 21)    # number of processes
t_s, t_w = 5e-6, 1e-10         # latency (s/message), per-word time (s/word)
peak = 1e12                    # 1 Tflop/s per process

comp = 2 * n**2 / (p * peak)
comm_1d = t_s * np.log2(p) + t_w * n
comm_2d = 2 * (t_s * np.log2(p) + t_w * n / np.sqrt(p))

plt.figure(figsize=(7, 4.5))
plt.loglog(p, comp, 'k--', label='computation only')
plt.loglog(p, comp + comm_1d, 'o-', label='1D blocking')
plt.loglog(p, comp + comm_2d, 's-', label='2D blocking')
plt.xlabel('number of processes $p$')
plt.ylabel('time per matvec (model)')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
print(f'communication floor, 1D: {t_w * n:.2e} s')
print(f'communication floor, 2D at p = {int(p[-1])}: {2 * t_s * np.log2(p[-1]) + 2 * t_w * n / np.sqrt(p[-1]):.2e} s')

### Packages for distributed storage

- [ScaLAPACK](http://www.netlib.org/scalapack/) -- dense linear algebra on distributed memory (block-cyclic layout); [SLATE](https://icl.utk.edu/slate/) is its modern successor.
- [Trilinos](https://trilinos.org/), [PETSc](https://petsc.org/) -- large sparse problems, solvers, preconditioners.
- In Python: [mpi4py](https://mpi4py.readthedocs.io/en/stable/) for your own parallel algorithms.
- PyTorch supports distributed training and data storage, see [the tutorial](https://pytorch.org/tutorials/intermediate/dist_tuto.html).

### Summary: large unstructured matrices

- Distributed storage, one block per process
- MPI: explicit messages, latency $t_s$ and bandwidth $1/t_w$ model
- 1D vs 2D blocking: the amount of communicated data, $n$ vs $n/\sqrt{p}$
- Ready-made packages: ScaLAPACK/SLATE, Trilinos, PETSc, mpi4py

## Sparse matrices: introduction

- For dense linear algebra we are limited by memory: $N^2$ parameters to store.
- The class of **sparse** matrices -- most elements are zero, only **nnz** (number of nonzeros) $\ll N^2$ numbers are stored -- at least makes storage possible.

The questions are:

- can we **solve linear systems** fast,
- can we **solve eigenvalue problems** fast

with sparse matrices?

## Plan for the sparse part

Now we talk about **sparse matrices**: where they arise, how we store them, how we operate with them.

- Formats: coordinate (COO), lists of lists (LIL), compressed sparse row (CSR); relation to graphs
- The matrix-by-vector product (SpMV)
- Parallel processing and locality: caches, reordering
- Fast direct solvers for Gaussian elimination (start): fill-in and orderings

## Applications of sparse matrices

Sparse matrices arise in:

- partial differential equations (PDE) -- mathematical modelling of physics;
- graph mining, e.g. social network analysis;
- recommender systems (the user-item matrix is sparse);
- machine learning: pruning of neural networks;
- wherever relations between objects are 'sparse'.

### Sparse matrices are ubiquitous in PDE

The simplest PDE (partial differential equation) is the **Poisson equation** for the temperature $u(x,y)$ in the square $\Omega = [0,1]^2$ with the heat source $f(x,y)$:

$$-\Delta u = -\Big(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2}\Big) = f(x,y), \quad x,y \in \Omega, \qquad u\big|_{\partial \Omega} = 0.$$

- $u$ = unknown field (temperature), $f$ = known source, $\partial \Omega$ = boundary of the square.
- (The Laplace equation $\Delta u = f$ gives exactly the **negative** of the same matrix.)

#### Discretization

- Put a grid with step $h = 1/(n+1)$, replace the second derivative by the finite difference:

$$-\frac{\partial^2 u}{\partial x^2} \approx -\frac{u(x+h) - 2u(x) + u(x-h)}{h^2} + \mathcal{O}(h^2),$$

same for $y$ -- and we get a **linear system**.
- First the **one-dimensional case**.

#### 1D case: a tridiagonal matrix

$$\frac{-u_{i-1} + 2u_i - u_{i+1}}{h^2} = f_i, \quad i = 1, \dots, N-1, \qquad u_0 = u_N = 0,$$

or $Au = f$ with the **symmetric positive definite** (SPD) matrix

$$A = \frac{1}{h^2}\begin{bmatrix} 2 & -1 & & & \\ -1 & 2 & -1 & & \\ & -1 & 2 & -1 & \\ & & -1 & 2 & -1 \\ & & & -1 & 2 \end{bmatrix} = \frac{1}{h^2}\,\mathrm{tridiag}(-1, 2, -1).$$

- SPD links directly to earlier lectures: Cholesky, conjugate gradients.

The matrix is **tridiagonal** (three nonzero diagonals) and **sparse**: $3N-2$ nonzeros instead of $N^2$.

It is also **Toeplitz** (constant along each diagonal) -- a structure we can compress further.

#### Block structure in 2D

In two dimensions the 5-point stencil gives

$$\frac{4u_{ij} - u_{(i-1)j} - u_{(i+1)j} - u_{i(j-1)} - u_{i(j+1)}}{h^2} = f_{ij},$$

or, in the **Kronecker product form**,

$$A = \frac{1}{h^2}\big(L_1 \otimes I + I \otimes L_1\big),$$

where $L_1 = \mathrm{tridiag}(-1, 2, -1)$ is the 1D Laplacian, $I$ is the identity, $\otimes$ is the Kronecker product, and $L_1 \otimes I + I \otimes L_1$ is called a **Kronecker sum**.

For $A \in \mathbb{R}^{n \times m}$ and $B \in \mathbb{R}^{l \times k}$ the Kronecker product is the block matrix

$$A \otimes B = \begin{bmatrix}a_{11}B & \dots & a_{1m}B \\ \vdots & \ddots & \vdots \\ a_{n1}B & \dots & a_{nm}B\end{bmatrix} \in \mathbb{R}^{nl \times mk}, \qquad (A \otimes B)_{(i-1)l+p,\ (j-1)k+q} = a_{ij} b_{pq}.$$

#### Block matrix form of the 2D Laplacian

$$A = \frac{1}{h^2}\begin{bmatrix} L_1 + 2I & -I & & & \\ -I & L_1 + 2I & -I & & \\ & -I & L_1 + 2I & -I & \\ & & -I & L_1 + 2I & -I \\ & & & -I & L_1 + 2I \end{bmatrix}$$

- Block-tridiagonal: $n \times n$ blocks; each diagonal block $L_1 + 2I = \mathrm{tridiag}(-1, 4, -1)$ is the 1D stencil -- exactly the 5-point stencil $4u_{ij} - \ldots$.

##### Kronecker product: short list of properties

- Bilinear: $(\alpha A + \beta B) \otimes C = \alpha A \otimes C + \beta B \otimes C$
- $(A \otimes B)(C \otimes D) = AC \otimes BD$ (when the products are defined)
- $(A \otimes B)^{-1} = A^{-1} \otimes B^{-1}$; eigenvalues of $A \otimes B$ are all products $\lambda_i(A) \lambda_j(B)$
- Let $\mathrm{vec}(X)$ be the **vectorization** of $X$ (stacking the columns). Then
$\mathrm{vec}(AXB) = (B^T \otimes A)\,\mathrm{vec}(X)$.
- Practical point: never form $A \otimes B$ explicitly if you can **apply** it implicitly -- the Kronecker sum matvec is two 1D passes over an $n \times n$ array of values.

## Demo: Kronecker algebra

- Check the vec identity numerically.
- Check the eigenvalue rule.
- Look at the sparsity of the Kronecker sum $L_1 \otimes I + I \otimes L_1$.

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# 1) vec identity:  vec(A X B) = (B^T kron A) vec(X)
m, n_, k_, l = 2, 3, 2, 4
A_ = rng.standard_normal((m, n_))
X = rng.standard_normal((n_, k_))
B_ = rng.standard_normal((k_, l))
lhs = (A_ @ X @ B_).reshape(-1, order='F')
rhs = np.kron(B_.T, A_) @ X.reshape(-1, order='F')
print('vec identity error:      ', np.max(np.abs(lhs - rhs)))

# 2) eigenvalues of a Kronecker product are products of eigenvalues
S1 = np.array([[2.0, 1.0], [1.0, 3.0]])
S2 = np.array([[1.0, -1.0], [-1.0, 5.0]])
ev = np.sort(np.linalg.eigvals(np.kron(S1, S2)).real)
ev_ref = np.sort(np.multiply.outer(np.linalg.eigvalsh(S1), np.linalg.eigvalsh(S2)).ravel())
print('Kronecker eigenvalues:   ', np.max(np.abs(ev - ev_ref)))

# 3) the 2D Laplacian is a Kronecker SUM (sparsity: 5-point stencil)
n = 5
L1 = sp.spdiags(np.vstack([-np.ones(n), 2 * np.ones(n), -np.ones(n)]), [-1, 0, 1], n, n).toarray()
A2d = np.kron(L1, np.eye(n)) + np.kron(np.eye(n), L1)
plt.figure(figsize=(4, 4))
plt.spy(A2d)
plt.title('$L_1 \\otimes I + I \\otimes L_1$: block tridiagonal')
plt.tight_layout()

### Sparse matrices in graph theory

- A graph is represented by its **adjacency matrix** ($a_{ij} = 1$ if there is an edge $i \to j$), which is usually sparse.
- Numerical solutions of graph problems = processing of this sparse matrix:
    - community detection and graph clustering
    - ranking (PageRank and friends)
    - random walks
- Example: one of the largest public hyperlink graphs -- 3.5 billion web pages, 128 billion hyperlinks, see [Web Data Commons](http://webdatacommons.org/hyperlinkgraph/).
- Medium-scale test graphs: [Stanford Large Network Dataset Collection (SNAP)](https://snap.stanford.edu/data/).

### SuiteSparse Matrix Collection

- Formerly the University of Florida Sparse Matrix Collection.
- More than 3000 matrices from all applications, with statistics: [sparse.tamu.edu](https://sparse.tamu.edu/).
- This is the standard benchmark zoo for sparse algorithms: always check your solver on it.

In [ ]:
from IPython.display import IFrame
IFrame('http://yifanhu.net/GALLERY/GRAPHS/search.html', width=700, height=450)   # needs internet

### Sparse matrices and deep learning

- A DNN/LLM has a lot of parameters, and many of them are redundant.
- **Pruning**: remove the weights with small magnitude, retrain -- smaller and faster models almost without accuracy loss (see [sparse variational dropout](https://github.com/ars-ashuha/variational-dropout-sparsifies-dnn), Han et al. 'Deep Compression').
- Hardware is getting involved: NVIDIA Tensor Cores support **2:4 structured sparsity** (two zeros out of four) since Ampere.
- The catch: **unstructured** sparsity gives sparse matrices that hardware handles poorly (see the efficiency slide) -- structured sparsity is much easier to speed up.
- Same story for inference: sparse attention, sparse activations.

## Sparse matrices: construction in SciPy

- We create sparse matrices with **scipy.sparse** (good, though not the only option: PETSc, Eigen, Intel MKL, cuSPARSE for GPUs).
- Modern SciPy distinguishes `csr_matrix` (subclass of `np.matrix`) and the preferred `csr_array` (subclass of `np.ndarray`).
- Useful functions:
    - `spdiags` / `diags` -- create from diagonals,
    - `kron` -- Kronecker product of sparse matrices,
    - overloaded arithmetic: $A + B$, $A @ x$, $A @ B$.
- Sizes are limited by memory for **nnz**, not $N^2$.

## Demo: build the 2D Laplacian

- One `kron` call, and we get a matrix with $N = n^2$ unknowns.
- `spy` plots the sparsity pattern: pixel $(i,j)$ is drawn if $a_{ij} \neq 0$.

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

n = 512                          # n x n grid  ->  N = n^2 unknowns
N = n * n
ex = np.ones(n)
L1 = sp.spdiags(np.vstack([-ex, 2 * ex, -ex]), [-1, 0, 1], n, n).tocsr()
I = sp.eye(n, format='csr')
A = sp.csr_array(sp.kron(L1, I) + sp.kron(I, L1))   # divide by h**2 for the true FD matrix

print(f'shape: {A.shape}, nnz = {A.nnz}, dense would need {N * N} numbers')

plt.figure(figsize=(5, 5))
plt.spy(A, marker=',', markersize=0.5)
plt.title(f'2D Laplacian, N = {N}')
plt.tight_layout()

### Sparsity pattern

- `spy` draws the **sparsity pattern** of the matrix.
- The pattern is what determines the complexity of sparse algorithms -- often it is all you need to estimate 'how hard' the matrix is.
- Examples: banded (good locality), block diagonal (independent subproblems), arrow (nasty for Gaussian elimination -- we will see why).

### Sparse matrix: definition

- A matrix is **sparse** if **nnz** (the number of nonzero elements) is much less than $N^2$ -- typically nnz $= \mathcal{O}(N)$.
- Then the basic linear algebra operations -- above all, solving $Ax = b$ -- can be done faster than with the full matrix.
- Complexity is measured in **nnz**, not in $N^2$.

## What we need to know to see how it works

- **Question 1:** how to store a sparse matrix in memory?
- **Question 2:** how to multiply a sparse matrix by a vector fast?
- **Question 3:** how to solve linear systems with sparse matrices fast?

### Sparse matrix storage

Many formats exist; the important ones:

- **COO** -- coordinate format
- **LIL** -- list of lists
- **CSR** -- compressed sparse row
- **CSC** -- compressed sparse column (CSR with rows and columns swapped)
- **DIA / ELL / BSR** -- block and diagonal variants (good for GPUs)

In `scipy` there is a constructor for each, e.g. `sp.csr_matrix(A)` or `sp.lil_matrix(A)`.

#### Coordinate format (COO)

The simplest format: positions and values of the nonzero elements,

```python
i, j, val
```

where `i`, `j` are integer arrays of indices, `val` is the array of values.

- We store $2 \cdot \mathbf{nnz}$ integers + $\mathbf{nnz}$ values.

**Q:** what is good and what is bad in this format?

#### Main disadvantages of COO

- **Storage**: two integers per value -- with 8-byte values and 4-byte indices that is 2 extra numbers per nonzero.
- **Matvec**: no fast access to 'the next element of row $i$' -- you have to scan (or sort) all entries; not cache-friendly.
- **Editing**: finding one element costs $\mathcal{O}(\mathbf{nnz})$ operations.

- The first two are fixed by **CSR**; the third by **LIL** (one Python list of column indices per row -- nice for building a matrix, wasteful for arithmetic).

#### Compressed sparse row (CSR)

In the CSR format a matrix is stored as 3 arrays:

```python
ia, ja, sa
```

- **ia** (row starts) -- integer array of length $N+1$: row $i$ occupies positions `ia[i] : ia[i+1]`;
- **ja** (column indices) -- integer array of length **nnz**;
- **sa** (values) -- real array of length **nnz**.

<img src="https://www.karlrupp.net/wp-content/uploads/2016/02/csr_storage_sparse_marix.png" width=55% />

Total: $2 \cdot \mathbf{nnz} + N + 1$ numbers.

## Demo: anatomy of CSR

- A tiny $4 \times 4$ matrix: print `ia`, `ja`, `sa`, and draw them.
- Compare the memory of the formats on a larger random matrix.

In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

dense = np.array([[1, 0, 0, 2],
                  [0, 0, 3, 0],
                  [4, 0, 0, 5],
                  [0, 6, 0, 0]])
A = sp.csr_matrix(dense)
print('ia =', A.indptr, '   (length', len(A.indptr), ')')
print('ja =', A.indices, '   (length', len(A.indices), ')')
print('sa =', A.data, '   (length', len(A.data), ')')

fig, axes = plt.subplots(3, 1, figsize=(7, 3.8), sharex=True)
for ax, arr, name in zip(axes, [A.indptr, A.indices, A.data],
                         ['ia: row starts', 'ja: column indices', 'sa: values']):
    ax.bar(np.arange(len(arr)), arr, color='steelblue')
    ax.set_ylabel(name)
plt.xlabel('index')
plt.tight_layout()

# memory of the formats for a bigger sparse matrix
B = sp.random(500, 500, density=0.01, format='csr', random_state=0)
for fmt in ('coo', 'csr', 'csc', 'lil', 'dia'):
    C = B.asformat(fmt)
    n_bytes = sum(v.nbytes for v in C.__dict__.values() if isinstance(v, np.ndarray))
    print(f'{fmt}: {n_bytes / 1024:8.1f} KiB')
print('(LIL hides its lists of Python ints -- the real overhead is even larger)')

### Sparse tensors in PyTorch and TensorFlow

- PyTorch supports COO (`torch.sparse_coo_tensor`) and since ~1.10 also CSR/CSC/BSR (`torch.sparse_csr_tensor`); `torch.sparse.mm` for products.
- Autograd support is still **incomplete**: many ops have no sparse backward pass, see [pytorch#9674](https://github.com/pytorch/pytorch/issues/9674).
- TensorFlow: `tf.sparse` in COO format, [supported ops](https://www.tensorflow.org/api_docs/python/tf/sparse); gradients limited as well.
- Rule of thumb: build sparse structures in SciPy, do the differentiable part dense or with special kernels.

### CSR and the sparse matvec (SpMV)

```python
for i in range(N):
    for k in range(ia[i], ia[i + 1]):
        y[i] += sa[k] * x[ja[k]]
```

- Cost: one multiplication and one addition per nonzero, $\approx 2 \cdot \mathbf{nnz}$ flops.
- Memory traffic: `sa` and `ja` are read sequentially (good), `x[ja[k]]` is read **randomly** (bad!), `y` is written sequentially (good).
- The result is **memory bound**: the time is set by memory traffic, not by the processor.

Let us do a short timing test of the formats.

In [ ]:
import time
import numpy as np
import scipy.sparse as sp

n = 256
N = n * n
ex = np.ones(n)
L1 = sp.spdiags(np.vstack([-ex, 2 * ex, -ex]), [-1, 0, 1], n, n).tocsr()
I = sp.eye(n, format='csr')
A = sp.csr_array(sp.kron(L1, I) + sp.kron(I, L1))
x = np.random.default_rng(0).standard_normal(N)

def bench(M, x, reps=20):
    M @ x                                            # warm up
    t0 = time.perf_counter()
    for _ in range(reps):
        y = M @ x
    return (time.perf_counter() - t0) / reps

print(f'nnz = {A.nnz}')
for name in ('csr', 'csc', 'coo', 'lil', 'dia'):
    M = A.asformat(name)
    t = bench(M, x)
    print(f'{name}: {t * 1e3:7.2f} ms   {2 * A.nnz / t / 1e9:6.2f} Gflops')
print('naive formats convert internally on every call -- measurable overhead')

### Sparse matrices and efficiency

- Sparse matrices give **complexity reduction**: $\mathcal{O}(\mathbf{nnz})$ instead of $\mathcal{O}(N^2)$.
- But they are **not very good** for parallel/GPU implementations: performance is limited by **random memory access**.
- Typically a peak efficiency of $10\%-15\%$ is considered good for SpMV.
- Dense kernels reach $> 80\%$; sparse kernels are bound by memory bandwidth and latency of scattered reads.

### Recall: how we measure efficiency

- **flop** = one floating point operation; **Gflops** = $10^9$ flops per second.
- Peak rate of a CPU: (cores) $\times$ (frequency) $\times$ (SIMD width) $\times 2$ (multiply + add).
- The standard way to measure the efficiency of linear algebra on a particular architecture is a **dense matvec** (or matmul) -- it runs at high efficiency and shows the peak.

In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)
n, k = 4000, 1400
A = rng.standard_normal((n, n))
V = rng.standard_normal((n, k))

t0 = time.perf_counter()
A @ V
t = time.perf_counter() - t0
print(f'time: {t:.2e} s, {2 * n * n * k / t / 1e9:.1f} Gflops')

In [ ]:
import numpy as np
import scipy.sparse as sp
import time

rng = np.random.default_rng(0)
n, k = 40_000, 500        # tridiagonal matrix, k right-hand sides
ex = np.ones(n)
A = sp.csr_array(sp.spdiags(np.vstack([-ex, 2 * ex, -ex]), [-1, 0, 1], n, n).tocsr())
R = rng.standard_normal((n, k))

t0 = time.perf_counter()
A @ R
t = time.perf_counter() - t0
print(f'nnz = {A.nnz}, time: {t:.2e} s, {2 * A.nnz * k / t / 1e9:.1f} Gflops')
print('compare with the dense number above: sparse works at a few percent of the peak')

### Random data access and cache misses

- Matrix and vector entries live in RAM (random access memory).
- To compute the matvec, the used parts are moved to the **cache** -- fast, small memory sitting on the CPU die (see the lecture about the Strassen algorithm).
- The CPU reads operands from the cache and writes the results back.
- If the CPU needs data that is not in the cache, this is a **cache miss** -- the data is moved from RAM to cache (hundreds of cycles).

**Q**: what if the cache is full? (answer: some old line is evicted -- say, the least recently used)

- The more cache misses, the slower the computation.

#### Cache analysis of the CSR matvec

```python
for i in range(N):
    for k in range(ia[i], ia[i + 1]):
        y[i] += sa[k] * x[ja[k]]
```

- `sa[k]`, `ja[k]`: sequential reads -- the prefetcher is happy.
- `x[ja[k]]`: **random** reads -- every nonzero may touch a different cache line.
- `y[i]`: sequential writes.

**Q**: which part of the operands causes the cache misses?
**Q**: how can this be fixed? (answer below)

### Reordering reduces cache misses

- If `ja` stores nearby indices, then the corresponding pieces of $x$ are moved to the cache together and the number of misses drops.
- This happens when the matrix is **banded** or at least **block diagonal**.
- We can convert a given matrix to a banded/block form by **permutations**:

- Let $P$ (rows) and $Q$ (columns) be permutation matrices, $A_1 = PAQ$.
- $y = Ax$ is equivalent to $\tilde{y} = A_1 \tilde{x}$, where $\tilde{x} = Q^T x$, $\tilde{y} = Py$ -- same flops, different access pattern.
- Typical tools: **RCM** (reverse Cuthill--McKee: minimizes bandwidth), minimum degree orderings.
- [Separated block diagonal format](http://albert-jan.yzelman.net/PDFs/yzelman09-rev.pdf) is a cache-oblivious format for SpMV; extensible to 2D (separate rows and columns).

## Demo: locality is not in the nnz

- The same tridiagonal matrix, three patterns: banded, randomly permuted, RCM-reordered.
- Same nnz, same flops -- different time. The pattern is what matters.

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import reverse_cuthill_mckee

rng = np.random.default_rng(1)
n = 5000
e = np.ones(n)
A_banded = sp.csr_array(sp.spdiags(np.vstack([-e, 2 * e, -e]), [-1, 0, 1], n, n).tocsr())

perm = rng.permutation(n)
A_perm = A_banded[perm][:, perm].tocsr()

r = reverse_cuthill_mckee(sp.csc_matrix(A_perm), symmetric_mode=True)
A_rcm = A_perm[r][:, r].tocsr()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, M, title in zip(axes, [A_banded, A_perm, A_rcm],
                        ['original (banded)', 'randomly permuted', 'RCM reordered']):
    ax.spy(M, marker=',', markersize=1)
    ax.set_title(title)
plt.tight_layout()

x = rng.standard_normal(n)
def bench(M, reps=100):
    M @ x
    t0 = time.perf_counter()
    for _ in range(reps):
        y = M @ x
    return (time.perf_counter() - t0) / reps

for name, M in [('banded', A_banded), ('permuted', A_perm), ('RCM', A_rcm)]:
    print(f'{name}: {bench(M) * 1e6:8.1f} us')
print('same nnz =', A_banded.nnz, 'for all three (the difference grows with n)')

### Sparse transpose matvec

- Often we need not only $Ax$ for sparse $A$, but also $A^T x$ -- e.g. in Krylov methods for non-symmetric systems (next lectures: GMRES/BiCGSTAB).
- In CSR the transpose is not free: the row traversal becomes a column traversal with scattered writes.
- [This paper](https://people.eecs.berkeley.edu/~aydin/csb2009.pdf) proposes the **compressed sparse block** (CSB) format, good for both $Ax$ and $A^T x$.

#### Compressed sparse block (CSB)

- Split the matrix into $b \times b$ blocks.
- Store block indices + indices of data **inside** each block -- a few bits per index, not 32/64.
- The order of blocks (and inside blocks) matters for parallel implementations.
- Switching between block-row and block-column traversal makes the same structure serve both $Ax$ and $A^T x$.

## Solving linear systems with sparse matrices

- **Direct methods**: sparse LU (SuperLU, UMFPACK, MUMPS, PARDISO) with reorderings to minimize **fill-in** -- today's last topic.
- **Krylov methods**: iterative, only matvecs with $A$ (and maybe $A^T$) + a preconditioner -- the next lectures.

## Demo: the Poisson equation, solved

- Solve the 2D Poisson system with `spsolve` (sparse LU).
- Look at the solution -- and at the amount of **fill-in** in the factors.

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

n = 256
N = n * n
ex = np.ones(n)
L1 = sp.spdiags(np.vstack([-ex, 2 * ex, -ex]), [-1, 0, 1], n, n).tocsr()
I = sp.eye(n, format='csr')
A = sp.csc_matrix(sp.kron(L1, I) + sp.kron(I, L1))
rhs = np.ones(N)

t0 = time.perf_counter()
sol = spla.spsolve(A, rhs)
t = time.perf_counter() - t0
print(f'N = {N}, solved in {t:.2f} s')

lu = spla.splu(A)
print(f'nnz(A) = {A.nnz},  nnz(L) + nnz(U) = {lu.L.nnz + lu.U.nnz}')

_, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(sol[::n + 1])
ax1.set_title('solution on the diagonal slice')
im = ax2.contourf(sol.reshape(n, n), levels=20)
ax2.set_title('solution of the Poisson equation')
plt.colorbar(im, ax=ax2)
plt.tight_layout()

### Fill-in: why sparse LU is not automatically cheap

- **Fill-in**: nonzeros that were zeros in $A$ but appear in $L$ and $U$ during Gaussian elimination.
- A sparse $A$ may have **dense** $L+U$ -- then the method is dense in disguise.
- The fill-in depends on the **ordering** of rows/columns:
    - natural ordering: fine for banded matrices;
    - **RCM**: keeps the band thin ($\mathcal{O}(N^{3/2})$ fill on a 2D grid);
    - **nested dissection / minimum degree** (COLAMD, MMD): $\mathcal{O}(N \log N)$ fill on a 2D grid -- the basis of modern 'fast direct solvers' (multifrontal methods).

## Demo: fill-in counterexample

- An **arrow matrix**: very sparse, but its LU is almost dense.
- The 2D Laplacian: fill-in depends strongly on the ordering.

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# 1) arrow matrix: sparse A, dense LU
m = 300
rows = np.concatenate([np.zeros(m, dtype=int), np.arange(m)])
cols = np.concatenate([np.arange(m), np.zeros(m, dtype=int)])
vals = np.ones(2 * m)
A_arrow = sp.csc_matrix(sp.coo_matrix((vals, (rows, cols)), shape=(m, m)))
print(f'arrow matrix: nnz(A) = {A_arrow.nnz}   (sparse!)')
lu = spla.splu(A_arrow)
print(f'but nnz(L) + nnz(U) = {lu.L.nnz + lu.U.nnz}   (dense would be {m * m})')

# 2) 2D Laplacian: the ordering decides the fill-in
n = 64
N = n * n
ex = np.ones(n)
L1 = sp.spdiags(np.vstack([-ex, 2 * ex, -ex]), [-1, 0, 1], n, n).tocsr()
I = sp.eye(n, format='csr')
A = sp.csc_matrix(sp.kron(L1, I) + sp.kron(I, L1))

print(f'2D Laplacian: N = {N}, nnz(A) = {A.nnz}')
for ordering in ('NATURAL', 'COLAMD', 'MMD_AT_PLUS_A'):
    lu = spla.splu(A, permc_spec=ordering)
    print(f'{ordering:14s}: nnz(L) + nnz(U) = {lu.L.nnz + lu.U.nnz}')

## Take home message

- Parallel matvec: 1D vs 2D blocking -- the amount of communicated data, $n$ vs $n/\sqrt{p}$; latency floors exist.
- Sparse formats: CSR (`ia`, `ja`, `sa`), $2 \cdot \mathbf{nnz} + N + 1$ numbers; pick the format for the operation.
- SpMV is memory-bound: locality and reordering (RCM, minimum degree) decide the speed, not the flop count.
- Direct solvers: the enemy is **fill-in**; orderings turn 'sparse but hopeless' into 'sparse and fast'.

# Questions?